In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: file_upload_utility (parameterized)
# Ingestion config JSON ko landing volume me (re)generate karta hai.
# target_table / checkpoint / source_path ab env config_setup se derive hote hain.
import json

config = {
  "rds_cdc": {
    "source_path": f"{LANDING}/rds/*/*.json",
    "target_table": f"{BRONZE}.rds_cdc",
    "format": "json",
    "checkpoint": f"{CHECKPOINTS}/rds_cdc",
    "schema_hints": "op STRING, _change_type STRING, _commit_ts STRING, _lsn LONG, table STRING, before STRING, after STRING",
    "dq_rules": {
      "critical": [
        {"name": "valid_lsn", "condition": "_lsn IS NOT NULL"},
        {"name": "valid_change_type", "condition": "_change_type IS NOT NULL"},
        {"name": "valid_table", "condition": "table IS NOT NULL"}
      ],
      "warn": [
        {"name": "valid_op", "condition": "op IN ('C','U','D')"}
      ]
    },
    "partition_by": "table"
  },
  "sap_incremental": {
    "source_path": f"{LANDING}/sap/incremental/*/",
    "target_table": f"{BRONZE}.sap_incremental",
    "format": "parquet",
    "checkpoint": f"{CHECKPOINTS}/sap_incremental",
    "schema_hints": "asset_id STRING, machine_id STRING, book_value DOUBLE, last_modified_ts STRING",
    "dq_rules": {
      "critical": [
        {"name": "valid_asset_id", "condition": "asset_id IS NOT NULL"},
        {"name": "valid_machine_id", "condition": "machine_id IS NOT NULL"}
      ],
      "warn": [
        {"name": "positive_book_value", "condition": "book_value > 0"}
      ]
    }
  },
  "telemetry": {
    "source_path": f"{LANDING}/telemetry/*/",
    "target_table": f"{BRONZE}.telemetry",
    "format": "json",
    "checkpoint": f"{CHECKPOINTS}/telemetry",
    "schema_hints": "event_id STRING, machine_id STRING, sensor_ts STRING, engine_temp DOUBLE, hydraulic_press DOUBLE, fuel_level DOUBLE, rpm LONG",
    "dq_rules": {
      "critical": [
        {"name": "valid_machine_id", "condition": "machine_id IS NOT NULL"},
        {"name": "valid_event_id", "condition": "event_id IS NOT NULL"}
      ],
      "warn": [
        {"name": "engine_temp_range", "condition": "engine_temp BETWEEN 0 AND 200"},
        {"name": "hydraulic_range", "condition": "hydraulic_press BETWEEN 0 AND 500"}
      ]
    }
  },
  "gps": {
    "source_path": f"{LANDING}/gps/*/",
    "target_table": f"{BRONZE}.gps",
    "format": "csv",
    "checkpoint": f"{CHECKPOINTS}/gps",
    "schema_hints": "ping_id STRING, machine_id STRING, site_id STRING, ping_ts STRING, lat DOUBLE, lon DOUBLE, speed DOUBLE",
    "csv_options": {"header": "true", "inferSchema": "true"},
    "dq_rules": {
      "critical": [
        {"name": "valid_ping_id", "condition": "ping_id IS NOT NULL"},
        {"name": "valid_machine_id", "condition": "machine_id IS NOT NULL"}
      ],
      "warn": [
        {"name": "valid_lat", "condition": "lat BETWEEN -90 AND 90"},
        {"name": "valid_lon", "condition": "lon BETWEEN -180 AND 180"}
      ]
    }
  },
  "kafka": {
    "source_path": f"{LANDING}/kafka/live_sensor_stream/",
    "target_table": f"{BRONZE}.kafka_stream",
    "format": "json",
    "checkpoint": f"{CHECKPOINTS}/kafka",
    "schema_hints": "stream_event_id STRING, machine_id STRING, sensor_ts STRING, payload_json STRING, schema_version LONG",
    "dq_rules": {
      "critical": [
        {"name": "valid_stream_id", "condition": "stream_event_id IS NOT NULL"},
        {"name": "valid_machine_id", "condition": "machine_id IS NOT NULL"}
      ],
      "warn": [
        {"name": "valid_payload", "condition": "payload_json IS NOT NULL AND length(payload_json) > 0"}
      ]
    }
  },
  "api": {
    "source_path": f"{LANDING}/api/*/",
    "target_table": f"{BRONZE}.api",
    "format": "json",
    "checkpoint": f"{CHECKPOINTS}/api",
    "schema_hints": "weather_id STRING, site_id STRING, obs_ts STRING, temp_c DOUBLE, humidity LONG, wind_kph DOUBLE, condition STRING",
    "dq_rules": {
      "critical": [
        {"name": "valid_site_id", "condition": "site_id IS NOT NULL"}
      ],
      "warn": [
        {"name": "temp_range", "condition": "temp_c BETWEEN -50 AND 60"},
        {"name": "humidity_range", "condition": "humidity BETWEEN 0 AND 100"}
      ]
    }
  },
  "ba_standardization": {
    "source_path": f"{LANDING}/ba/",
    "target_table": f"{BRONZE}.ba_location_standardized",
    "format": "csv",
    "checkpoint": f"{CHECKPOINTS}/ba",
    "schema_hints": "raw_city STRING, raw_country STRING, std_city STRING, std_state STRING, std_country STRING, confidence DOUBLE",
    "csv_options": {"header": "true", "inferSchema": "true"},
    "dq_rules": {
      "critical": [
        {"name": "valid_raw_city", "condition": "raw_city IS NOT NULL"},
        {"name": "valid_std_city", "condition": "std_city IS NOT NULL"},
        {"name": "valid_confidence", "condition": "confidence BETWEEN 0 AND 1"}
      ],
      "warn": [
        {"name": "high_confidence", "condition": "confidence >= 0.7"}
      ]
    }
  }
}

# Create _config folder + write file
dbutils.fs.mkdirs(f"{LANDING}/_config")
with open(INGESTION_CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Config written to {INGESTION_CONFIG_PATH}")

display(dbutils.fs.ls(f"{LANDING}/_config/"))